In [3]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(
    "./processor"
)

tokenizer = processor.tokenizer

vocab = tokenizer.get_vocab()

labels = [
    token
    for token, idx in sorted(
        vocab.items(),
        key=lambda x: x[1]
    )
]

print("Vocabulary size:", len(labels))
print("Labels:", labels)

Vocabulary size: 36
Labels: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'á', 'í', 'ó', 'ú', 'ı', 'ń', 'ǵ', '|', '[UNK]', '[PAD]']


In [4]:
print("PAD token ID:", tokenizer.pad_token_id)
print("Word delimiter:", tokenizer.word_delimiter_token)

PAD token ID: 35
Word delimiter: |


In [5]:
import os

kenlm_path = "/home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.arpa"

print("KenLM mavjud:", os.path.isfile(kenlm_path))
print("KenLM hajmi:", round(os.path.getsize(kenlm_path) / 1024**2, 2), "MB")

KenLM mavjud: True
KenLM hajmi: 89.39 MB


In [6]:
from pyctcdecode import build_ctcdecoder

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=kenlm_path,
    alpha=0.5,
    beta=1.0
)

print("✅ KenLM decoder tayyor!")

✅ KenLM decoder tayyor!


Loading the LM will be faster if you build a binary file.
Reading /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


In [9]:
import os

BASE_DIR="/home/intern/projects/karakalpak_stt"

print("BASE_DIR mavjud:", os.path.isdir(BASE_DIR))
print("Joriy papka:", os.getcwd())

for folder in ["data", "models", "checkpoints", "results"]:

    path = os.path.join(BASE_DIR, folder)

    print(f"\n {folder.upper()}:")
    print("Manzil:", path)

    if os.path.isdir(path):
        for item in os.listdir(path):
            print("  ", item)
    else:
        print("Papka topilmadi")

BASE_DIR mavjud: True
Joriy papka: /home/intern/projects/karakalpak_stt

 DATA:
Manzil: /home/intern/projects/karakalpak_stt/data
   val_metadata.parquet
   test_metadata.parquet
   train_metadata.parquet

 MODELS:
Manzil: /home/intern/projects/karakalpak_stt/models
   karakalpak_xlsr300m_lora_ctc_v1
   final_karakalpak_stt

 CHECKPOINTS:
Manzil: /home/intern/projects/karakalpak_stt/checkpoints
   full_lora_step_1250
   full_lora_step_3750
   full_lora_step_3000
   xlsr_lora_bf16_30steps
   xlsr_lora_step_1000
   full_lora_step_6000
   full_lora_step_8500
   xlsr_lora_step_200
   full_lora_step_2500
   full_lora_step_3500
   full_lora_step_11000
   full_lora_step_12500
   full_lora_step_4250
   full_lora_step_8000
   full_lora_step_5000
   xlsr_lora_step_100
   full_lora_step_13000
   full_lora_step_3250
   full_lora_step_1750
   full_lora_step_11500
   full_lora_step_13023
   full_lora_step_9000
   full_lora_step_4000
   xlsr_lora_step_500
   full_lora_step_2000
   full_lora_step_100


In [12]:
from pathlib import Path
import pandas as pd

BASE = Path("/home/intern/projects/karakalpak_stt")

for model_name in [
    "karakalpak_xlsr300m_lora_ctc_v1",
    "final_karakalpak_stt"
]:
    path = BASE / "models" / model_name

    print(f"\nMODEL: {model_name}")

    for file in path.rglob("*"):
        if file.is_file():
            print(" ", file.relative_to(path))

val_path = BASE / "data" / "val_metadata.parquet"

val_df = pd.read_parquet(val_path)

print("\nVALIDATION DATASET")
print("Shape:", val_df.shape)
print("Columns:", val_df.columns.tolist())
print(val_df.head(3).to_string(index=False, max_colwidth=80))

result_path = BASE / "results" / "kenlm_validation_comparison.csv"

if result_path.exists():
    comparison = pd.read_csv(result_path)

    print("\nKENLM VALIDATION RESULTS")
    print(comparison.to_string(index=False))


MODEL: karakalpak_xlsr300m_lora_ctc_v1
  README.md
  adapter_config.json
  adapter_model.safetensors
  model_metadata.json
  processor/tokenizer_config.json
  processor/processor_config.json
  processor/vocab.json

MODEL: final_karakalpak_stt

VALIDATION DATASET
Shape: (965, 4)
Columns: ['id', 'cleaned_text_lowercase', 'source_file', 'source_row']
   id                                                           cleaned_text_lowercase       source_file  source_row
18716 sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yu... train_019.parquet         659
18364 konferenciyaǵa qaraqalpaqstan respublikası ministrler keńesi baslıǵınıń máslá... train_019.parquet         311
11793 sol sebepli qanday kontent kóbirek kóriledi degen soraw ornına qanday kontent... train_012.parquet         768

KENLM VALIDATION RESULTS
  Decoder       WER       CER
   Greedy 49.889618 11.525404
KenLM_WER 26.904097  9.950211
KenLM_CER 27.705993  9.083989


In [14]:
import json
from pathlib import Path

BASE = Path("/home/intern/projects/karakalpak_stt")

adapter_path = BASE / "models" / "karakalpak_xlsr300m_lora_ctc_v1"

config_path = adapter_path / "adapter_config.json"

with open(config_path, "r") as f:
    config = json.load(f)

print("BASE MODEL:")
print(config.get("base_model_name_or_path"))

print("\nTARGET MODULES:")
print(config.get("target_modules"))

print("\nLORA RANK:")
print(config.get("r"))

print("\nLORA ALPHA:")
print(config.get("lora_alpha"))

print("\nMODULES TO SAVE:")
print(config.get("modules_to_save"))

BASE MODEL:
facebook/wav2vec2-xls-r-300m

TARGET MODULES:
['v_proj', 'q_proj']

LORA RANK:
8

LORA ALPHA:
16

MODULES TO SAVE:
['lm_head']


In [15]:
import torch

from pathlib import Path
from transformers import Wav2Vec2ForCTC,Wav2Vec2Processor
from peft import PeftModel

BASE=Path("/home/intern/projects/karakalpak_stt")
BASE_MODEL="facebook/wav2vec2-xls-r-300m"

ADAPTER_PATH=(BASE/"models" / "karakalpak_xlsr300m_lora_ctc_v1")

PROCESSOR_PATH=BASE/"processor"

processor=Wav2Vec2Processor.from_pretrained(str(PROCESSOR_PATH))

base_model=Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    vocab_size=len(processor.tokenizer),
    pad_token_id=processor.tokenizer.pad_token_id,
    ignore_mismatched_sizes=True
)

model=PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_PATH)
)

device="cuda" if torch.cuda.is_available() else "cpu"

model=model.to(device)
model.eval()

print("Model yuklandi")
print("Device:",device)
print("Vocabulary size:",model.config.vocab_size)
print("CTC head:",model.lm_head)

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model yuklandi
Device: cuda
Vocabulary size: 36
CTC head: ModulesToSaveWrapper(
  (original_module): Linear(in_features=1024, out_features=36, bias=True)
  (modules_to_save): ModuleDict(
    (default): Linear(in_features=1024, out_features=36, bias=True)
  )
)


In [17]:
from pathlib import Path
import pandas as pd

BASE = Path("/home/intern/projects/karakalpak_stt")

audio_dir = BASE / "audio_shards"

print("AUDIO SHARDS:")

files = sorted(audio_dir.rglob("*.parquet"))

print("Jami parquet fayllar:", len(files))

for file in files[:10]:
    print(file.relative_to(BASE))

val_df = pd.read_parquet(
    BASE / "data" / "val_metadata.parquet"
)

sample = val_df.iloc[0]

print("\nVALIDATION SAMPLE:")
print(sample.to_dict())

if files:
    shard_df = pd.read_parquet(files[0])

    print("\nSHARD COLUMNS:")
    print(shard_df.columns.tolist())

    print("\nSHARD SHAPE:")
    print(shard_df.shape)

    print("\nAUDIO TYPE:")
    if "audio" in shard_df.columns:
        print(type(shard_df.iloc[0]["audio"]))

AUDIO SHARDS:
Jami parquet fayllar: 20
audio_shards/train_001.parquet
audio_shards/train_002.parquet
audio_shards/train_003.parquet
audio_shards/train_004.parquet
audio_shards/train_005.parquet
audio_shards/train_006.parquet
audio_shards/train_007.parquet
audio_shards/train_008.parquet
audio_shards/train_009.parquet
audio_shards/train_010.parquet

VALIDATION SAMPLE:
{'id': 18716, 'cleaned_text_lowercase': 'sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwdi rawajlandırıw boyınsha pikir alıstı', 'source_file': 'train_019.parquet', 'source_row': 659}

SHARD COLUMNS:
['id', 'audio', 'cleaned_text', 'cleaned_text_lowercase']

SHARD SHAPE:
(991, 4)

AUDIO TYPE:
<class 'bytes'>


In [18]:
import io
import torch
import soundfile as sf
import librosa
import pandas as pd

from pathlib import Path

BASE = Path("/home/intern/projects/karakalpak_stt")

# 1. Validation metadata
val_df = pd.read_parquet(
    BASE / "data" / "val_metadata.parquet"
)

sample = val_df.iloc[0]

# 2. Audio joylashgan parquet
shard_path = BASE / "audio_shards" / sample["source_file"]

shard_df = pd.read_parquet(shard_path)

# 3. Audio bytes
audio_bytes = shard_df.iloc[int(sample["source_row"])]["audio"]

# 4. Bytes -> waveform
audio, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

# Stereo bo‘lsa mono qilamiz
if audio.ndim == 2:
    audio = audio.mean(axis=1)

# 16 kHz ga o‘tkazamiz
if sr != 16000:
    audio = librosa.resample(
        audio,
        orig_sr=sr,
        target_sr=16000
    )
    sr = 16000

print("Audio shape:", audio.shape)
print("Sample rate:", sr)
print("Duration:", len(audio) / sr)

# 5. Processor
inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs.input_values.to(device)

# 6. XLS-R + LoRA + CTC
model.eval()

with torch.inference_mode():
    logits = model(
        input_values
    ).logits

print("Logits shape:", logits.shape)

# 7. Greedy decoding
predicted_ids = torch.argmax(
    logits,
    dim=-1
)

prediction = processor.batch_decode(
    predicted_ids.cpu()
)[0]

print("\nREFERENCE:")
print(sample["cleaned_text_lowercase"])

print("\nPREDICTION:")
print(prediction)

Audio shape: (236374,)
Sample rate: 16000
Duration: 14.773375
Logits shape: torch.Size([1, 738, 36])

REFERENCE:
sondayaq delegaciya aǵzaları ilajlarda belsene qatnasıp shbsh mámleketleri yuridikalıq bilimlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwdi rawajlandırıw boyınsha pikir alıstı

PREDICTION:
sondayaq delegaciya aǵzaları ilajlarda belsne qatnasıp shıbıshı mámleketleri yuridikalıq bilemlendiriw mákemeleri arasındaǵı ilimiy hám akademiyalıq birge islesiwde rawajlandırıw boyınsha pikir alıstı


In [19]:
print("Metadata ID:", sample["id"])
print("Audio shard ID:", shard_df.iloc[int(sample["source_row"])]["id"])

Metadata ID: 18716
Audio shard ID: 18716


In [20]:
import io
import gc
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch

from pathlib import Path
from tqdm.auto import tqdm
from jiwer import wer, cer

BASE = Path("/home/intern/projects/karakalpak_stt")

VAL_PATH = BASE / "data" / "val_metadata.parquet"
AUDIO_DIR = BASE / "audio_shards"

SAVE_DIR = BASE / "results" / "validation_logits"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

val_df = pd.read_parquet(VAL_PATH)

references = []
predictions = []
manifest = []

model.eval()

print("Validation samples:", len(val_df))

# Bir xil shardni qayta-qayta o'qimaslik uchun guruhlaymiz
groups = val_df.groupby("source_file", sort=False)

with torch.inference_mode():

    for source_file, group in tqdm(
        groups,
        total=len(groups),
        desc="Processing shards"
    ):

        shard_path = AUDIO_DIR / source_file
        shard_df = pd.read_parquet(shard_path)

        for _, sample in group.iterrows():

            row = int(sample["source_row"])

            audio_record = shard_df.iloc[row]

            # Audio va reference to'g'ri juftlanganini tekshirish
            assert int(audio_record["id"]) == int(sample["id"]), (
                f"ID mismatch: {source_file}, row={row}"
            )

            # Audio bytes -> waveform
            audio, sr = sf.read(
                io.BytesIO(audio_record["audio"]),
                dtype="float32"
            )

            if audio.ndim == 2:
                audio = audio.mean(axis=1)

            if sr != 16000:
                audio = librosa.resample(
                    audio,
                    orig_sr=sr,
                    target_sr=16000
                )

            inputs = processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt"
            )

            input_values = inputs.input_values.to(device)

            # Model inference
            outputs = model(input_values)

            logits = outputs.logits[0].float().cpu().numpy()

            # Diskka float16 formatida saqlaymiz
            logits_path = SAVE_DIR / f"{int(sample['id'])}.npy"

            np.save(
                logits_path,
                logits.astype(np.float16)
            )

            # Greedy decoding
            predicted_ids = np.argmax(logits, axis=-1)

            prediction = processor.decode(
                predicted_ids
            )

            reference = sample["cleaned_text_lowercase"]

            references.append(reference)
            predictions.append(prediction)

            manifest.append({
                "id": int(sample["id"]),
                "reference": reference,
                "prediction": prediction,
                "logits_path": str(logits_path)
            })

        del shard_df
        gc.collect()

# Natijalarni saqlash
results_df = pd.DataFrame(manifest)

results_df.to_csv(
    BASE / "results" / "validation_logits_manifest.csv",
    index=False
)

# WER / CER
greedy_wer = wer(references, predictions) * 100
greedy_cer = cer(references, predictions) * 100

print("\n========== RESULTS ==========")
print("Samples:", len(references))
print(f"Greedy WER: {greedy_wer:.4f}%")
print(f"Greedy CER: {greedy_cer:.4f}%")
print("Logits folder:", SAVE_DIR)

Validation samples: 965


Processing shards:   0%|          | 0/20 [00:00<?, ?it/s]


========== RESULTS ==========
Samples: 965
Greedy WER: 49.9383%
Greedy CER: 11.5193%
Logits folder: /home/intern/projects/karakalpak_stt/results/validation_logits


In [21]:
import numpy as np
import pandas as pd

from pathlib import Path
from itertools import product
from tqdm.auto import tqdm
from jiwer import wer, cer
from pyctcdecode import build_ctcdecoder

BASE = Path("/home/intern/projects/karakalpak_stt")

MANIFEST_PATH = (
    BASE / "results" / "validation_logits_manifest.csv"
)

KENLM_PATH = (
    BASE / "kenlm" / "karakalpak_5gram.arpa"
)

RESULT_PATH = (
    BASE / "results" / "kenlm_alpha_beta_tuning.csv"
)

# 1. Validation metadata
manifest = pd.read_csv(MANIFEST_PATH)

references = manifest["reference"].astype(str).tolist()

# 2. Oldindan hisoblangan logitslarni yuklash
validation_logits = [
    np.load(path).astype(np.float32)
    for path in tqdm(
        manifest["logits_path"],
        desc="Loading logits"
    )
]

assert len(validation_logits) == len(references) == 965

assert all(
    x.ndim == 2 and x.shape[1] == len(labels)
    for x in validation_logits
)

print("Validation samples:", len(references))

# 3. Decoder
decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(KENLM_PATH),
    alpha=0.5,
    beta=1.0
)

# 4. Parameter combinations
alphas = [0.2, 0.4, 0.6, 0.8, 1.0]
betas = [0.0, 0.5, 1.0, 1.5, 2.0]

combinations = list(product(alphas, betas))

print("Total experiments:", len(combinations))

results = []

# 5. Alpha/Beta tuning
for alpha, beta in tqdm(
    combinations,
    desc="KenLM tuning"
):

    decoder.reset_params(
        alpha=alpha,
        beta=beta
    )

    predictions = []

    for logits in validation_logits:

        prediction = decoder.decode(
            logits,
            beam_width=100
        )

        predictions.append(prediction)

    current_wer = wer(
        references,
        predictions
    ) * 100

    current_cer = cer(
        references,
        predictions
    ) * 100

    results.append({
        "alpha": alpha,
        "beta": beta,
        "beam_width": 100,
        "WER": current_wer,
        "CER": current_cer
    })

    print(
        f"Alpha={alpha:.1f} | "
        f"Beta={beta:.1f} | "
        f"WER={current_wer:.4f}% | "
        f"CER={current_cer:.4f}%"
    )

    # Har bir tajribadan keyin saqlash
    pd.DataFrame(results).to_csv(
        RESULT_PATH,
        index=False
    )

# 6. Eng yaxshi natijalar
df = pd.DataFrame(results)

df = df.sort_values("WER").reset_index(drop=True)

print("\n========== TOP 10 ==========")
print(df.head(10).to_string(index=False))

print("\n========== BEST PARAMETERS ==========")

best = df.iloc[0]

print("Best Alpha:", best["alpha"])
print("Best Beta:", best["beta"])
print("Best Beam Width:", int(best["beam_width"]))
print(f"Best WER: {best['WER']:.4f}%")
print(f"Best CER: {best['CER']:.4f}%")

Loading logits:   0%|          | 0/965 [00:00<?, ?it/s]

Validation samples: 965
Total experiments: 25


Loading the LM will be faster if you build a binary file.
Reading /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


KenLM tuning:   0%|          | 0/25 [00:00<?, ?it/s]

Alpha=0.2 | Beta=0.0 | WER=24.7581% | CER=7.6883%
Alpha=0.2 | Beta=0.5 | WER=25.2516% | CER=7.6899%
Alpha=0.2 | Beta=1.0 | WER=25.8522% | CER=7.7149%
Alpha=0.2 | Beta=1.5 | WER=26.7645% | CER=7.7452%
Alpha=0.2 | Beta=2.0 | WER=27.9040% | CER=7.7883%
Alpha=0.4 | Beta=0.0 | WER=22.0473% | CER=7.6464%
Alpha=0.4 | Beta=0.5 | WER=22.1706% | CER=7.5895%
Alpha=0.4 | Beta=1.0 | WER=22.4044% | CER=7.5353%
Alpha=0.4 | Beta=1.5 | WER=22.6966% | CER=7.4930%
Alpha=0.4 | Beta=2.0 | WER=22.9823% | CER=7.4573%
Alpha=0.6 | Beta=0.0 | WER=22.0473% | CER=8.1147%
Alpha=0.6 | Beta=0.5 | WER=21.7583% | CER=7.9486%
Alpha=0.6 | Beta=1.0 | WER=21.5019% | CER=7.7822%
Alpha=0.6 | Beta=1.5 | WER=21.6122% | CER=7.7253%
Alpha=0.6 | Beta=2.0 | WER=21.6577% | CER=7.6626%
Alpha=0.8 | Beta=0.0 | WER=24.4400% | CER=8.9490%
Alpha=0.8 | Beta=0.5 | WER=23.6056% | CER=8.7380%
Alpha=0.8 | Beta=1.0 | WER=22.9401% | CER=8.5054%
Alpha=0.8 | Beta=1.5 | WER=22.3882% | CER=8.3116%
Alpha=0.8 | Beta=2.0 | WER=21.9564% | CER=8.1340%


In [23]:
import numpy as np
import pandas as pd

from pathlib import Path
from itertools import product
from tqdm.auto import tqdm
from jiwer import wer, cer
from pyctcdecode import build_ctcdecoder

BASE = Path("/home/intern/projects/karakalpak_stt")

MANIFEST_PATH = BASE / "results" / "validation_logits_manifest.csv"

KENLM_PATH = BASE / "kenlm" / "karakalpak_5gram.arpa"

RESULT_PATH = BASE / "results" / "kenlm_fine_grained_tuning.csv"

# 1. Validation metadata
manifest = pd.read_csv(MANIFEST_PATH)

references = manifest["reference"].astype(str).tolist()

# 2. Logitslarni yuklash
validation_logits = [
    np.load(path).astype(np.float32)
    for path in tqdm(
        manifest["logits_path"],
        desc="Loading logits"
    )
]

print("Validation samples:", len(references))

# 3. KenLM decoder
decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(KENLM_PATH),
    alpha=0.6,
    beta=1.0
)

# 4. Yangi parametrlar
alphas = [0.50, 0.55, 0.60, 0.65, 0.70]
betas = [0.50, 0.75, 1.00, 1.25, 1.50]

combinations = list(product(alphas, betas))

print("Experiments:", len(combinations))

results = []

# 5. Fine-grained tuning
for alpha, beta in tqdm(
    combinations,
    desc="Fine-grained tuning"
):

    decoder.reset_params(
        alpha=alpha,
        beta=beta
    )

    predictions = [
        decoder.decode(
            logits,
            beam_width=100
        )
        for logits in validation_logits
    ]

    current_wer = wer(references, predictions) * 100
    current_cer = cer(references, predictions) * 100

    results.append({
        "alpha": alpha,
        "beta": beta,
        "beam_width": 100,
        "WER": current_wer,
        "CER": current_cer
    })

    print(
        f"Alpha={alpha:.2f} | "
        f"Beta={beta:.2f} | "
        f"WER={current_wer:.4f}% | "
        f"CER={current_cer:.4f}%"
    )

    # Natijalarni saqlash
    pd.DataFrame(results).to_csv(
        RESULT_PATH,
        index=False
    )

# 6. Eng yaxshi natija
df = pd.DataFrame(results)

df = df.sort_values(
    ["WER", "CER"]
).reset_index(drop=True)

print("\n========== TOP 10 ==========")
print(df.head(10).to_string(index=False))

best = df.iloc[0]

print("\n========== BEST CONFIG ==========")

print("Alpha:", best["alpha"])
print("Beta:", best["beta"])
print("Beam Width:", int(best["beam_width"]))
print(f"WER: {best['WER']:.4f}%")
print(f"CER: {best['CER']:.4f}%")

Loading logits:   0%|          | 0/965 [00:00<?, ?it/s]

Validation samples: 965
Experiments: 25


Loading the LM will be faster if you build a binary file.
Reading /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Fine-grained tuning:   0%|          | 0/25 [00:00<?, ?it/s]

Alpha=0.50 | Beta=0.50 | WER=21.7681% | CER=7.7172%
Alpha=0.50 | Beta=0.75 | WER=21.7843% | CER=7.6803%
Alpha=0.50 | Beta=1.00 | WER=21.9012% | CER=7.6580%
Alpha=0.50 | Beta=1.25 | WER=21.9336% | CER=7.6457%
Alpha=0.50 | Beta=1.50 | WER=22.0245% | CER=7.6426%
Alpha=0.55 | Beta=0.50 | WER=21.5960% | CER=7.7902%
Alpha=0.55 | Beta=0.75 | WER=21.6187% | CER=7.7541%
Alpha=0.55 | Beta=1.00 | WER=21.6350% | CER=7.7091%
Alpha=0.55 | Beta=1.25 | WER=21.6901% | CER=7.6864%
Alpha=0.55 | Beta=1.50 | WER=21.7616% | CER=7.6718%
Alpha=0.60 | Beta=0.50 | WER=21.7583% | CER=7.9486%
Alpha=0.60 | Beta=0.75 | WER=21.5798% | CER=7.8525%
Alpha=0.60 | Beta=1.00 | WER=21.5019% | CER=7.7822%
Alpha=0.60 | Beta=1.25 | WER=21.5603% | CER=7.7668%
Alpha=0.60 | Beta=1.50 | WER=21.6122% | CER=7.7253%
Alpha=0.65 | Beta=0.50 | WER=22.0278% | CER=8.1128%
Alpha=0.65 | Beta=0.75 | WER=21.7681% | CER=8.0263%
Alpha=0.65 | Beta=1.00 | WER=21.7453% | CER=7.9694%
Alpha=0.65 | Beta=1.25 | WER=21.5798% | CER=7.8794%
Alpha=0.65 |

In [25]:
import time
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from jiwer import wer, cer
from pyctcdecode import build_ctcdecoder

BASE = Path("/home/intern/projects/karakalpak_stt")

MANIFEST_PATH = BASE / "results" / "validation_logits_manifest.csv"
KENLM_PATH = BASE / "kenlm" / "karakalpak_5gram.arpa"

RESULT_PATH = BASE / "results" / "kenlm_beam_width_tuning.csv"

manifest = pd.read_csv(MANIFEST_PATH)
references = manifest["reference"].astype(str).tolist()

validation_logits = [
    np.load(path).astype(np.float32)
    for path in tqdm(
        manifest["logits_path"],
        desc="Loading logits"
    )
]

alpha = 0.65
beta = 1.50

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(KENLM_PATH),
    alpha=alpha,
    beta=beta
)

beam_widths = [25, 50, 100, 150, 200]

results = []

for beam_width in tqdm(
    beam_widths,
    desc="Beam Width tuning"
):

    start_time = time.perf_counter()

    predictions = [
        decoder.decode(
            logits,
            beam_width=beam_width
        )
        for logits in validation_logits
    ]

    elapsed = time.perf_counter() - start_time

    current_wer = wer(references, predictions) * 100
    current_cer = cer(references, predictions) * 100

    results.append({
        "alpha": alpha,
        "beta": beta,
        "beam_width": beam_width,
        "WER": current_wer,
        "CER": current_cer,
        "total_seconds": elapsed,
        "seconds_per_audio": elapsed / len(references)
    })

    pd.DataFrame(results).to_csv(
        RESULT_PATH,
        index=False
    )

    print(
        f"Beam={beam_width:3d} | "
        f"WER={current_wer:.4f}% | "
        f"CER={current_cer:.4f}% | "
        f"Time={elapsed:.1f}s"
    )

df_beam = pd.DataFrame(results)
df_beam = df_beam.sort_values(
    ["WER", "CER", "seconds_per_audio"]
).reset_index(drop=True)

print("\n========== BEAM RESULTS ==========")
print(df_beam.to_string(index=False))

best = df_beam.iloc[0]

print("\n========== BEST CONFIG ==========")
print("Alpha:", best["alpha"])
print("Beta:", best["beta"])
print("Beam Width:", int(best["beam_width"]))
print(f"WER: {best['WER']:.4f}%")
print(f"CER: {best['CER']:.4f}%")
print(f"Seconds/audio: {best['seconds_per_audio']:.3f}")

Loading logits:   0%|          | 0/965 [00:00<?, ?it/s]

Loading the LM will be faster if you build a binary file.
Reading /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Beam Width tuning:   0%|          | 0/5 [00:00<?, ?it/s]

Beam= 25 | WER=23.9010% | CER=8.8595% | Time=33.4s
Beam= 50 | WER=22.2940% | CER=8.1851% | Time=50.3s
Beam=100 | WER=21.4791% | CER=7.8129% | Time=75.8s
Beam=150 | WER=21.1642% | CER=7.6330% | Time=97.4s
Beam=200 | WER=21.0149% | CER=7.5645% | Time=120.8s

========== BEAM RESULTS ==========
 alpha  beta  beam_width       WER      CER  total_seconds  seconds_per_audio
  0.65   1.5         200 21.014869 7.564544     120.798395           0.125180
  0.65   1.5         150 21.164210 7.632980      97.358223           0.100889
  0.65   1.5         100 21.479125 7.812915      75.820661           0.078571
  0.65   1.5          50 22.294007 8.185086      50.291740           0.052116
  0.65   1.5          25 23.901045 8.859455      33.353046           0.034563

========== BEST CONFIG ==========
Alpha: 0.65
Beta: 1.5
Beam Width: 200
WER: 21.0149%
CER: 7.5645%
Seconds/audio: 0.125


In [27]:
BASE = Path("/home/intern/projects/karakalpak_stt")

TEST_PATH = BASE / "data" / "test_metadata.parquet"
AUDIO_DIR = BASE / "audio_shards"

RESULTS_DIR = BASE / "results"
RESULTS_DIR.mkdir(exist_ok=True)

test_df = pd.read_parquet(TEST_PATH)

print("Test samples:", len(test_df))

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(
        BASE / "kenlm" / "karakalpak_5gram.arpa"
    ),
    alpha=0.65,
    beta=1.50
)

beam_width = 200

model.eval()

results = []

start_time = time.perf_counter()

with torch.inference_mode():

    for source_file, group in tqdm(
        test_df.groupby("source_file", sort=False),
        desc="Test evaluation"
    ):

        shard_df = pd.read_parquet(
            AUDIO_DIR / source_file
        )

        for _, sample in group.iterrows():

            row = int(sample["source_row"])

            audio_record = shard_df.iloc[row]

            assert int(audio_record["id"]) == int(sample["id"])

            audio, sr = sf.read(
                io.BytesIO(audio_record["audio"]),
                dtype="float32"
            )

            if audio.ndim == 2:
                audio = audio.mean(axis=1)

            if sr != 16000:
                audio = librosa.resample(
                    audio,
                    orig_sr=sr,
                    target_sr=16000
                )

            inputs = processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt"
            )

            input_values = inputs.input_values.to(device)

            logits = model(input_values).logits

            logits = logits[0].float().cpu().numpy()

            greedy_ids = np.argmax(logits, axis=-1)

            greedy_prediction = processor.decode(greedy_ids)

            kenlm_prediction = decoder.decode(
                logits,
                beam_width=beam_width
            )

            results.append({
                "id": int(sample["id"]),
                "reference": sample["cleaned_text_lowercase"],
                "greedy_prediction": greedy_prediction,
                "kenlm_prediction": kenlm_prediction
            })

elapsed = time.perf_counter() - start_time

df = pd.DataFrame(results)

references = df["reference"].tolist()

greedy_predictions = df["greedy_prediction"].tolist()
kenlm_predictions = df["kenlm_prediction"].tolist()

greedy_wer = wer(references, greedy_predictions) * 100
greedy_cer = cer(references, greedy_predictions) * 100

kenlm_wer = wer(references, kenlm_predictions) * 100
kenlm_cer = cer(references, kenlm_predictions) * 100

print("\n========== FINAL TEST RESULTS ==========")

print("Test samples:", len(df))

print(f"Greedy WER: {greedy_wer:.4f}%")
print(f"Greedy CER: {greedy_cer:.4f}%")

print(f"KenLM WER: {kenlm_wer:.4f}%")
print(f"KenLM CER: {kenlm_cer:.4f}%")

print(f"Evaluation time: {elapsed:.2f}s")

df.to_csv(
    RESULTS_DIR / "optimized_kenlm_test_predictions.csv",
    index=False
)

pd.DataFrame([{
    "alpha": 0.65,
    "beta": 1.50,
    "beam_width": 200,
    "greedy_wer": greedy_wer,
    "greedy_cer": greedy_cer,
    "kenlm_wer": kenlm_wer,
    "kenlm_cer": kenlm_cer,
    "evaluation_seconds": elapsed
}]).to_csv(
    RESULTS_DIR / "optimized_kenlm_test_metrics.csv",
    index=False
)

print("\n✅ Test natijalari saqlandi!")

Test samples: 965


Loading the LM will be faster if you build a binary file.
Reading /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Test evaluation:   0%|          | 0/20 [00:00<?, ?it/s]


========== FINAL TEST RESULTS ==========
Test samples: 965
Greedy WER: 49.1943%
Greedy CER: 11.5740%
KenLM WER: 21.5938%
KenLM CER: 7.7890%
Evaluation time: 218.99s

✅ Test natijalari saqlandi!


In [29]:
BASE = Path("/home/intern/projects/karakalpak_stt")

df = pd.read_csv(
    BASE / "results" / "optimized_kenlm_test_predictions.csv"
)

references = df["reference"].astype(str).tolist()
predictions = df["kenlm_prediction"].astype(str).tolist()

result = process_words(references, predictions)

print("========== ERROR ANALYSIS ==========")

print(f"WER: {result.wer * 100:.4f}%")
print("Substitutions:", result.substitutions)
print("Deletions:", result.deletions)
print("Insertions:", result.insertions)
print("Correct words:", result.hits)

total_errors = (
    result.substitutions +
    result.deletions +
    result.insertions
)

print("\n========== ERROR DISTRIBUTION ==========")

for name, count in [
    ("Substitutions", result.substitutions),
    ("Deletions", result.deletions),
    ("Insertions", result.insertions)
]:
    print(f"{name}: {count / total_errors * 100:.2f}%")

========== ERROR ANALYSIS ==========
WER: 21.5938%
Substitutions: 4993
Deletions: 1190
Insertions: 464
Correct words: 24599

========== ERROR DISTRIBUTION ==========
Substitutions: 75.12%
Deletions: 17.90%
Insertions: 6.98%


In [ ]:
BASE = Path("/home/intern/projects/karakalpak_stt")

df = pd.read_csv(
    BASE / "results" / "optimized_kenlm_test_predictions.csv"
)

substitutions = Counter()
deletions = Counter()
insertions = Counter()

for _, row in df.iterrows():

    ref = str(row["reference"])
    hyp = str(row["kenlm_prediction"])

    result = process_words(ref, hyp)

    ref_words = result.references[0]
    hyp_words = result.hypotheses[0]

    for chunk in result.alignments[0]:

        if chunk.type == "substitute":

            ref_chunk = ref_words[
                chunk.ref_start_idx:chunk.ref_end_idx
            ]

            hyp_chunk = hyp_words[
                chunk.hyp_start_idx:chunk.hyp_end_idx
            ]

            substitutions[(
                " ".join(ref_chunk),
                " ".join(hyp_chunk)
            )] += 1

        elif chunk.type == "delete":

            for word in ref_words[
                chunk.ref_start_idx:chunk.ref_end_idx
            ]:
                deletions[word] += 1

        elif chunk.type == "insert":

            for word in hyp_words[
                chunk.hyp_start_idx:chunk.hyp_end_idx
            ]:
                insertions[word] += 1

print("\n========== TOP 30 SUBSTITUTIONS ==========")

for (reference, prediction), count in substitutions.most_common(30):
    print(
        f"{reference:25s} → "
        f"{prediction:25s} | "
        f"{count}"
    )

print("\n========== TOP 20 DELETIONS ==========")

for word, count in deletions.most_common(20):
    print(f"{word:30s} | {count}")

print("\n========== TOP 20 INSERTIONS ==========")

for word, count in insertions.most_common(20):
    print(f"{word:30s} | {count}")


========== TOP 30 SUBSTITUTIONS ==========
haqqında                  → haqqındaǵı                | 21
jıyırma                   → jigirma                   | 9
atırǵanın                 → atırǵan                   | 5
sanaat                    → sanat                     | 5
qırq                      → qırıq                     | 4
rawajlanıwı               → rawajlanıw                | 4
atırǵanı                  → atırǵan                   | 4
ótilgenindey              → de                        | 4
altı                      → altınshı                  | 4
joqarǵı                   → joqarı                    | 4
hám                       → menen                     | 3
elli                      → eliw                      | 3
jılı                      → jıl                       | 3
ornında                   → orında                    | 3
sondayaq                  → sonday                    | 3
islesiwdi                 → islesiw                   | 3
bes                       →

In [31]:
BASE = Path("/home/intern/projects/karakalpak_stt")

df = pd.read_csv(
    BASE / "results" / "optimized_kenlm_test_predictions.csv"
)

records = []

for _, row in df.iterrows():

    reference = str(row["reference"])
    greedy = str(row["greedy_prediction"])
    kenlm = str(row["kenlm_prediction"])

    greedy_wer = wer(reference, greedy) * 100
    kenlm_wer = wer(reference, kenlm) * 100

    records.append({
        "id": row["id"],
        "reference": reference,
        "greedy": greedy,
        "kenlm": kenlm,
        "greedy_wer": greedy_wer,
        "kenlm_wer": kenlm_wer,
        "improvement": greedy_wer - kenlm_wer
    })

analysis = pd.DataFrame(records)

improved = (analysis["improvement"] > 0).sum()
worsened = (analysis["improvement"] < 0).sum()
unchanged = (analysis["improvement"] == 0).sum()

print("========== GREEDY VS KENLM ==========")

print("Total audios:", len(analysis))
print("KenLM improved:", improved)
print("KenLM worsened:", worsened)
print("Unchanged:", unchanged)

print("\n========== TOP 10 IMPROVEMENTS ==========")

print(
    analysis.sort_values(
        "improvement",
        ascending=False
    )[[
        "id",
        "greedy_wer",
        "kenlm_wer",
        "improvement"
    ]].head(10).to_string(index=False)
)

print("\n========== TOP 10 WORST CASES ==========")

print(
    analysis.sort_values(
        "improvement"
    )[[
        "id",
        "greedy_wer",
        "kenlm_wer",
        "improvement"
    ]].head(10).to_string(index=False)
)

analysis.to_csv(
    BASE / "results" / "greedy_vs_kenlm_analysis.csv",
    index=False
)

========== GREEDY VS KENLM ==========
Total audios: 965
KenLM improved: 949
KenLM worsened: 5
Unchanged: 11

========== TOP 10 IMPROVEMENTS ==========
   id  greedy_wer  kenlm_wer  improvement
 4205   85.185185  14.814815    70.370370
 2164   80.000000  10.000000    70.000000
 6922   75.862069   6.896552    68.965517
 5060   63.157895   0.000000    63.157895
12966   68.750000   6.250000    62.500000
10840   87.500000  25.000000    62.500000
 3156   66.666667   4.166667    62.500000
14927   67.500000   7.500000    60.000000
16002   88.571429  31.428571    57.142857
16204   95.652174  39.130435    56.521739

========== TOP 10 WORST CASES ==========
   id  greedy_wer  kenlm_wer  improvement
13899   33.333333  44.444444   -11.111111
 5248   16.129032  22.580645    -6.451613
 4696   24.390244  29.268293    -4.878049
13799   12.500000  16.666667    -4.166667
 4502   58.823529  61.764706    -2.941176
14394   44.000000  44.000000     0.000000
 1303   31.578947  31.578947     0.000000
 6837   2

In [32]:
from pathlib import Path

BASE = Path("/home/intern/projects/karakalpak_stt")

CHECKPOINT_DIR = BASE / "checkpoints"

checkpoint_names = [
    "full_lora_step_1250",
    "full_lora_step_2500",
    "full_lora_step_3000",
    "full_lora_step_3750",
    "full_lora_step_6000",
    "full_lora_step_8500"
]

for name in checkpoint_names:

    path = CHECKPOINT_DIR / name

    print(f"\n{'=' * 55}")
    print("CHECKPOINT:", name)
    print(f"{'=' * 55}")

    if not path.exists():
        print("❌ Topilmadi")
        continue

    for file in sorted(path.rglob("*")):
        if file.is_file():
            size_mb = file.stat().st_size / (1024**2)

            print(
                f"{str(file.relative_to(path)):45s}"
                f"{size_mb:.2f} MB"
            )


CHECKPOINT: full_lora_step_1250
adapter_config.json                          0.00 MB
processor_config.json                        0.00 MB
tokenizer_config.json                        0.00 MB
training_state.pt                            9.63 MB
vocab.json                                   0.00 MB

CHECKPOINT: full_lora_step_2500
adapter_config.json                          0.00 MB
processor_config.json                        0.00 MB
tokenizer_config.json                        0.00 MB
training_state.pt                            9.72 MB
vocab.json                                   0.00 MB

CHECKPOINT: full_lora_step_3000
adapter_config.json                          0.00 MB
processor_config.json                        0.00 MB
tokenizer_config.json                        0.00 MB
training_state.pt                            9.75 MB
vocab.json                                   0.00 MB

CHECKPOINT: full_lora_step_3750
adapter_config.json                          0.00 MB
processor_config.jso

In [33]:
import torch
from pathlib import Path

BASE = Path("/home/intern/projects/karakalpak_stt")

checkpoint_path = (
    BASE
    / "checkpoints"
    / "full_lora_step_8500"
    / "training_state.pt"
)

# Faqat o'zimiz yaratgan checkpointni o'qiymiz.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("CHECKPOINT TYPE:")
print(type(checkpoint))

if isinstance(checkpoint, dict):

    print("\n========== CHECKPOINT KEYS ==========")

    for key, value in checkpoint.items():

        if isinstance(value, dict):
            print(
                f"{key}: dict "
                f"({len(value)} elements)"
            )

            print(
                "  First keys:",
                list(value.keys())[:10]
            )

        elif isinstance(value, torch.Tensor):
            print(
                f"{key}: Tensor "
                f"{tuple(value.shape)}"
            )

        else:
            print(
                f"{key}: {type(value).__name__}"
            )

CHECKPOINT TYPE:
<class 'dict'>

========== CHECKPOINT KEYS ==========
global_step: int
model_weights: dict (98 elements)
  First keys: ['base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_A.default.weight', 'base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_B.default.weight', 'base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_A.default.weight', 'base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_B.default.weight', 'base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_A.default.weight', 'base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_B.default.weight', 'base_model.model.wav2vec2.encoder.layers.1.attention.q_proj.lora_A.default.weight', 'base_model.model.wav2vec2.encoder.layers.1.attention.q_proj.lora_B.default.weight', 'base_model.model.wav2vec2.encoder.layers.2.attention.v_proj.lora_A.default.weight', 'base_model.model.wav2vec2.encoder.layers.2.attention.v_proj.lora_B.default.weight']
optimizer_sta

In [34]:
weights = checkpoint["model_weights"]

print("Total weights:", len(weights))

print("\n========== LAST 10 WEIGHTS ==========")

for name, tensor in list(weights.items())[-10:]:
    print(name, tuple(tensor.shape))

print("\n========== CTC HEAD ==========")

for name, tensor in weights.items():
    if "lm_head" in name:
        print(name, tuple(tensor.shape))

print("\n========== CURRENT MODEL MATCH ==========")

current_weights = model.state_dict()

matched = [
    name for name in weights
    if name in current_weights
]

missing = [
    name for name in weights
    if name not in current_weights
]

print("Matched:", len(matched))
print("Not matched:", len(missing))

if missing:
    print("\nNot matched keys:")
    for name in missing[:20]:
        print(name)

Total weights: 98

========== LAST 10 WEIGHTS ==========
base_model.model.wav2vec2.encoder.layers.22.attention.v_proj.lora_A.default.weight (8, 1024)
base_model.model.wav2vec2.encoder.layers.22.attention.v_proj.lora_B.default.weight (1024, 8)
base_model.model.wav2vec2.encoder.layers.22.attention.q_proj.lora_A.default.weight (8, 1024)
base_model.model.wav2vec2.encoder.layers.22.attention.q_proj.lora_B.default.weight (1024, 8)
base_model.model.wav2vec2.encoder.layers.23.attention.v_proj.lora_A.default.weight (8, 1024)
base_model.model.wav2vec2.encoder.layers.23.attention.v_proj.lora_B.default.weight (1024, 8)
base_model.model.wav2vec2.encoder.layers.23.attention.q_proj.lora_A.default.weight (8, 1024)
base_model.model.wav2vec2.encoder.layers.23.attention.q_proj.lora_B.default.weight (1024, 8)
base_model.model.lm_head.modules_to_save.default.weight (36, 1024)
base_model.model.lm_head.modules_to_save.default.bias (36,)

========== CTC HEAD ==========
base_model.model.lm_head.modules_to_save

In [35]:
BASE = Path("/home/intern/projects/karakalpak_stt")

CHECKPOINT_DIR = BASE / "checkpoints"
AUDIO_DIR = BASE / "audio_shards"
VAL_PATH = BASE / "data" / "val_metadata.parquet"

RESULT_PATH = BASE / "results" / "checkpoint_comparison.csv"

checkpoint_names = [
    "full_lora_step_1250",
    "full_lora_step_2500",
    "full_lora_step_3000",
    "full_lora_step_3750",
    "full_lora_step_6000",
    "full_lora_step_8500"
]

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(
        BASE / "kenlm" / "karakalpak_5gram.arpa"
    ),
    alpha=0.65,
    beta=1.50
)

val_df = pd.read_parquet(VAL_PATH)

references = val_df["cleaned_text_lowercase"].astype(str).tolist()

base_model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    vocab_size=36,
    pad_token_id=processor.tokenizer.pad_token_id,
    ignore_mismatched_sizes=True
)

eval_model = PeftModel.from_pretrained(
    base_model,
    str(BASE / "models" / "karakalpak_xlsr300m_lora_ctc_v1")
)

eval_model = eval_model.to(device)
eval_model.eval()

audio_cache = {}

print("Loading validation audio...")

for source_file, group in val_df.groupby("source_file"):

    shard_df = pd.read_parquet(AUDIO_DIR / source_file)

    for _, sample in group.iterrows():

        row = int(sample["source_row"])
        record = shard_df.iloc[row]

        assert int(record["id"]) == int(sample["id"])

        audio, sr = sf.read(
            io.BytesIO(record["audio"]),
            dtype="float32"
        )

        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        if sr != 16000:
            audio = librosa.resample(
                audio,
                orig_sr=sr,
                target_sr=16000
            )

        audio_cache[int(sample["id"])] = audio

    del shard_df
    gc.collect()

print("Loaded audios:", len(audio_cache))

results = []

for checkpoint_name in checkpoint_names:

    print(f"\n========== {checkpoint_name} ==========")

    start_time = time.perf_counter()

    state_path = (
        CHECKPOINT_DIR / checkpoint_name / "training_state.pt"
    )

    checkpoint = torch.load(
        state_path,
        map_location="cpu",
        weights_only=True
    )

    weights = checkpoint["model_weights"]

    current_state = eval_model.state_dict()

    for key, value in weights.items():
        assert key in current_state, f"Missing key: {key}"
        assert current_state[key].shape == value.shape, (
            f"Shape mismatch: {key}"
        )

    load_result = eval_model.load_state_dict(
        weights,
        strict=False
    )

    assert len(load_result.unexpected_keys) == 0

    eval_model.eval()

    predictions = []

    with torch.inference_mode():

        for _, sample in tqdm(
            val_df.iterrows(),
            total=len(val_df),
            desc=checkpoint_name
        ):

            audio = audio_cache[int(sample["id"])]

            inputs = processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt"
            )

            input_values = inputs.input_values.to(device)

            logits = eval_model(input_values).logits[0]

            logits = logits.float().cpu().numpy()

            prediction = decoder.decode(
                logits,
                beam_width=200
            )

            predictions.append(prediction)

    current_wer = wer(references, predictions) * 100
    current_cer = cer(references, predictions) * 100

    elapsed = time.perf_counter() - start_time

    results.append({
        "checkpoint": checkpoint_name,
        "global_step": checkpoint["global_step"],
        "WER": current_wer,
        "CER": current_cer,
        "seconds": elapsed
    })

    pd.DataFrame(results).to_csv(
        RESULT_PATH,
        index=False
    )

    print(f"WER: {current_wer:.4f}%")
    print(f"CER: {current_cer:.4f}%")
    print(f"Time: {elapsed:.1f}s")

    del checkpoint, weights
    gc.collect()

# Yakuniy jadval
df_results = pd.DataFrame(results)

df_results = df_results.sort_values("WER")

print("\n========== CHECKPOINT COMPARISON ==========")
print(df_results.to_string(index=False))

print("\n========== BEST CHECKPOINT ==========")
print(df_results.iloc[0].to_string())

Loading the LM will be faster if you build a binary file.
Reading /home/intern/projects/karakalpak_stt/kenlm/karakalpak_5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading validation audio...
Loaded audios: 965

========== full_lora_step_1250 ==========


full_lora_step_1250:   0%|          | 0/965 [00:00<?, ?it/s]

...

WER: 23.6511%
CER: 8.9287%
Time: 261.9s

========== full_lora_step_6000 ==========


full_lora_step_6000:   0%|          | 0/965 [00:00<?, ?it/s]

WER: 22.4044%
CER: 8.2274%
Time: 233.6s

========== full_lora_step_8500 ==========


full_lora_step_8500:   0%|          | 0/965 [00:00<?, ?it/s]

WER: 21.3428%
CER: 7.4907%
Time: 224.1s

========== CHECKPOINT COMPARISON ==========
         checkpoint  global_step       WER       CER    seconds
full_lora_step_8500         8500 21.342770  7.490725 224.061501
full_lora_step_6000         6000 22.404389  8.227378 233.629377
full_lora_step_3750         3750 23.651062  8.928661 261.850461
full_lora_step_3000         3000 24.118564  8.717199 266.939927
full_lora_step_2500         2500 25.027596  9.337742 271.208248
full_lora_step_1250         1250 28.585806 11.306253 316.655864

========== BEST CHECKPOINT ==========
checkpoint     full_lora_step_8500
global_step                   8500
WER                       21.34277
CER                       7.490725
seconds                 224.061501
